In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os
from IPython.display import JSON, display
import json
import math
from utils.classification_init import apply_style, visualize, compare_dicts

apply_style()

load_dotenv(dotenv_path='.env', override=True)
client = OpenAI()
[m.id for m in client.models.list() if "alias" not in m.id]

['15 - Apertus-8B-Instruct-2509 - A new swiss model from September 2025',
 '20 - EVE-Instruct - Expert Earth Observation and Earth Science (ES) domains',
 '01 - GPT-OSS-120b - an open model released by OpenAI in August 2025',
 '01 - MiniMax-M2.7 - our best model as of April, 2026',
 '02 - Qwen3.5-122B-A10B-FP8, general purpose large model',
 '09 - Qwen3-Coder-Next-FP8 from Feb 2026',
 'gpt-3.5-turbo',
 'text-davinci-003',
 'text-embedding-ada-002',
 '07 - Qwen3.5-35B-A3B - Multimodal model from Feb 2026',
 '999 - Mis',
 'eve-instruct-4gpu',
 'faster-whisper-large-v3',
 'nemotron-3-nano-omni-30b-bf16-262k-8gpu',
 '08 - Qwen3.6-35B-A3B-FP8 - Multimodal model from Apr 2026']

In [ ]:
model = "alias-fast"

# 🕵️ Analysing the Response Structure

Before continuing, take a moment to explore how OpenAI API responses are structured.<br>
Use the interactive JSON viewer from `display_response_as_json` and compare it with the visual overview created by `visualize`.<br>
This will help you understand how the model’s output is organized before you start working with it.

In [ ]:
def display_resonse_as_json(response):
    # Display formatted and collapsible JSON
    display(JSON(json.loads(response.model_dump_json())))

In [ ]:
message = "What scientific field is related to the concept of Entropy? Answer only with the scientific field, nothing else!"
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": message}],
    logprobs=True,
    top_logprobs = 5,
    temperature=1.0,
    max_tokens=20
)

In [ ]:
visualize(response)

In [ ]:
display_resonse_as_json(response)

### 🧩 **Inspecting the Response Data**

Use the **interactive JSON viewer** above to explore the structure of the API response.
Then, fill in the `<TODO>` fields in the code cell below to print:

* the **assistant’s message content**,
* the **log probability of the first token**, and
* the corresponding **token probability**.

This exercise helps you understand where key information (text and token-level data) is located in the response object.

In [ ]:
print("Response Id:", response.id)
print("Response Message:", "<TODO>")
print("Response First Token logprob:", "<TODO>")
print("Response First Token probability:", "<TODO>") # use math.exp

### Helper Function
The helper function `extract_probabilities` below retrieves information about the **alternative tokens** considered by the model at each token position.
You can access a specific token’s alternatives using:

```python
extract_probabilities(response)[token_index][alternative_index]
```

* `token_index` → the position of the token in the model’s response
* `alternative_index` → the index of an alternative token (the **chosen token** is always at index `0`, no matter its probability)

Run the code below and examine the output to understand how the probabilities are structured.
Then, compare these printed values with the visualization produced by `visualize(response)` to see how they correspond.

In [ ]:
def extract_probabilities(response):
    logprob_info_per_token = response.choices[0].logprobs.content
    return [
        [
            {
                "token": alternative.token,
                "logprob": alternative.logprob,
                "prob": math.exp(alternative.logprob)
            }
            for alternative in logprob_info.top_logprobs
        ]
        for logprob_info in logprob_info_per_token
    ]

In [ ]:
extract_probabilities(response)[0] # [0] -> first token

In [ ]:
extract_probabilities(response)[0][0]["prob"]

In [ ]:
visualize(response)

<br>
<br>

## 1️⃣ Using LLMs for Classification

### 🧩 Exercise - Classification of Paper Abstracts

In [ ]:
with open("science_abstracts.json", "r") as f:
    abstracts = json.loads(f.read())
display(JSON(abstracts))

In [ ]:
choices = {
    "A": "physics",
    "B": "math",
    "C": "chemistry",
    "D": "biology"
}
labels = list(choices.keys())
categories = list(choices.values())
choices_str = "\n".join([f"{label}: {category}" for label, category in choices.items()])
print(choices_str)

In [ ]:
# 🧩 TODO: Create Prompt
prompt = """
{abstract}
{choices}
"""

### Testing the Response for the First Abstract

In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": prompt.format(abstract=abstracts[10]["text"], choices=choices_str)}],
    logprobs=True,
    top_logprobs = 5,
    temperature=0,
    max_tokens=20
)

In [ ]:
visualize(response)

In [ ]:
extract_probabilities(response)[0]

The first token probabilities must now be filtered by tokens, that represent choices. Then, the highest ranking choice should be selected. If no alternative is present that represents one of the choices, return None. Implement the function `extract_class_probabilities` based on this description and try it out 

In [ ]:
# 🧩 TODO: Implement extract_most_probable_category
def extract_most_probable_category(response):
    first_token_alternatives = ... # see above
    filtered = ... # filter token alternatives for those where the token is one of the labels
    try:
        # get the alternative with the highest probability from the filtered list
        # -> with temperature=0 the list is ordered descending on token probability
        # return the category with the highest prob - "physics", "chemistry", "math", or "biology"

        return ...
    except:
        # return None if the filtered list is empty
        return None

In [ ]:
print("The expected category is", abstracts[0]["category"])
print("The model responded with category", extract_most_probable_category(response))

### Running Classification
After implementing the above function `extract_most_probable_category` you can now run these 2 cells to perform classification on all 20 abstracts and visualize the result in a confusion matrix.

In [ ]:
ground_truth = [a["category"] for a in abstracts]
predictions = []
for idx, abstract in enumerate(abstracts):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt.format(abstract=abstract["text"], choices=choices_str)}],
        logprobs=True,
        top_logprobs = 5,
        temperature=0,
        max_tokens=20
    )
    category = extract_most_probable_category(response)
    print(f"Abstract {idx+1} predicted to by in category {category}")
    predictions.append(category)

In [ ]:
# --- Print a Confusion Matrix given predicted and ground_truths ---
import matplotlib.pyplot as plt
import numpy as np

# Build confusion matrix manually
n = len(categories)
cm = np.zeros((n, n), dtype=int)

# Count missing predictions
none_count = sum(p is None for p in predictions)
if none_count > 0:
    print(f"⚠️ Warning: {none_count} predictions were None and were excluded from the plot.")

# Fill counts, skipping None
for true, pred in zip(ground_truth, predictions):
    if pred is None:
        continue
    i = categories.index(true)
    j = categories.index(pred)
    cm[i, j] += 1

# Plot
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm, cmap="Blues")

# Add counts
for i in range(n):
    for j in range(n):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black")

# Labels and ticks
ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))
ax.set_xticklabels(categories, rotation=45, ha="right")
ax.set_yticklabels(categories)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix" + ("(None values excluded)" if none_count > 0 else ""))

# Add colorbar
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 2️⃣ Using LLMs for Data Extraction - Structured Decoding

In structured decoding, the model’s token sampling process is constrained by a predefined output structure.
The OpenAI API Protocol supports this by allowing you to provide a JSON schema that defines the exact format of the desired output.
During decoding, the model is restricted to generating only those tokens that keep the response valid under the given schema.

Importantly, this schema is not part of the model’s prompt — it is enforced only during decoding.
This means the model itself receives no explicit instruction about the expected output structure, and fields such as "description" in the schema are not used for conditioning the model’s generation. As such, including information about the expected output format in the prompt is useful for increasing task comprehension and accuracy.

Below is a quick overview of the key properties of a JSON schema.
A schema is provided as a Python dictionary containing both a name and the actual schema definition under the "schema" key.

### 💡 Basic JSON Schema Properties (for all types)

* **`type`** – data type (`object`, `array`, `string`, `integer`, `number`, `boolean`)
* **`description`** – short explanation of the field’s meaning - optional
* **`enum`** – list of allowed values the property is restricted to

**For Objects** `{"type": "object"}`
* **`properties`** – defines allowed keys and their schemas
* **`required`** – list of mandatory keys

**For Arrays** `{"type": "array"}`

* **`items`** – schema for each array element
* **`minItems` / `maxItems`** – limit array length
* **`uniqueItems`** – require elements to be unique (`true`/`false`)

**For Strings** `{"type": "string"}`

* **`pattern`** – regex that the string must match
* **`minLength` / `maxLength`** – character length limits

**For Numbers / Integers** `{"type": "number"}`

* **`minimum` / `maximum`** – range constraints

### 💡Basic Example

In [ ]:
# Example input text
text = """George Orwell published 1984 in 1949, a dystopian novel about surveillance and control."""

# Define JSON schema
schema = {
    "name": "metadata_extraction",
    "schema": {
        "type": "object",
        "properties": {
            "title": {"type": "string"},
            "publication_year": {"type": "number"},
        }
    }
}

# Create the request
response = client.chat.completions.create(
    model=model,
    messages=[{
        "role": "user",
        "content": f"Extract the relevant information from the text - return only the :\n\n{text}"
    }],
    temperature=0.0, # <--- Important!
    response_format={"type": "json_schema", "json_schema": schema}
)
message = response.choices[0].message.content
json_message = json.loads(message)
print(json.dumps(json_message, indent=2))

### 🧩 Exercise - Extract Cinema Screenings
In this exercise, you’ll practice extracting structured information from unstructured text.
Your goal is to identify all **movie screenings** in the paragraph below and output them as structured JSON objects.

Each screening should include:

* 🎞️ **Cinema name**
* 📅 **Date** (use `DD.MM.YYYY` format)
* ⏰ **Time** (24h format)
* 🗣️ **Language** (e.g., "German" or "English")

#### Steps

1. **Define your JSON schema** in the cell below.
   * Use appropriate data types (`string`) and regex patterns to validate dates and times.
   * Make sure all four fields are included and set as *required*.
2. **Write your prompt** to instruct the model to extract all screenings from the text using that schema.
3. **Run your code** and check whether the model correctly returns all screenings.
4. **Compare your output** with the *expected result* in the final cell to self-validate your solution.

#### Example

```json
{
  "cinema_name": "Thalia Kino Dresden",
  "date": "12.05.2024",
  "time": "18:00",
  "language": "English"
}
```

*Hint:* Remember that using `pattern` with regex is a great way to ensure your date and time formats are consistent!

In [ ]:
screenings = """
The movie Oppenheimer returns to Dresden cinemas in May 2024.
The first screening takes place at Schauburg Dresden on Monday, May 5th, starting at 07:30pm, dubbed in German.
Later that week, Programmkino Ost will show the film on Wednesday, May 7th, at 8:15 pm, in the original version.
Finally, UFA Kristallpalast is hosting a late-night screening on 10.05.2024 at 21:00, also dubbed in English.
"""

In [ ]:
screening_extraction_schema = {
    "name": "screening_extraction",
    "schema": {
        <TODO>
    }
}

In [ ]:
prompt = """
<TODO Prompt>
{screenings}
"""
response = client.chat.completions.create(
    model=model,
    messages=[{
        "role": "user",
        "content": prompt.format(screenings=screenings)
    }],
    temperature=0.0,
    response_format={
        "type": "json_schema",
        "json_schema": screening_extraction_schema
    }
)
message = response.choices[0].message.content
json_message = json.loads(message)
print(json.dumps(json_message, indent=2))

You may use the following snippet to check whether the extracted data is correct. If it is, the function will print `✅ All values match.`.

In [ ]:
with open("screening_times.json", "r") as f:
    ground_truth = json.loads(f.read())

compare_dicts(ground_truth, json_message)

### [Extra] 🧩 Exercise - Create Random Numbers

In this task, you are supposed to create a schema that lets the model generate **27 random numbers**, each between **100 and 200**.

💡 *Hint:* Use 
* `"type": "array"` with `"minItems"` / `"maxItems"` for the length and optionally `"uniqueItems"` for unique random numbers
* `"type": "number"` with `"minimum"` / `"maximum"` for the range.

In [ ]:
random_number_schema = {
    "name": "random_number_generation",
    "schema": {
        <TODO>
    }
}

In [ ]:
prompt = "Generate random numbers between 100 and 200!"
response = client.chat.completions.create(
    model=model,
    messages=[{
        "role": "user",
        "content": prompt
    }],
    temperature=0.0,
    response_format={
        "type": "json_schema",
        "json_schema": random_number_schema
    }
)
message = response.choices[0].message.content
random_numbers = json.loads(message)
print("Random Numbers:", random_numbers)
print(f"Model responsded with {len(random_numbers)} random numbers!")

### [Extra] 🧩 Exercise - Extract Contact Data (Regex & Optional Fields)

In this task, you’ll work with another short text — this time about **contact details**.
Your goal is to extract the relevant information in a clear, structured format.

Think about what kind of data should be captured and how to ensure the **correct format** for each field.

#### Steps

1. Design an appropriate **JSON schema** for the task.
   * Consider what fields are always present and which might be *optional*.
   * Use **patterns** where helpful to validate specific formats (e.g., emails, phone numbers).
2. Write a short prompt instructing the model to extract all contact information from the text.
3. Run your code and inspect the result.
4. Compare your output with the *expected result* in the final cell.

*Tip:* This task focuses on using **regex** and **optional fields** — try to make your schema both strict enough to validate structure and flexible enough to handle variations.

In [ ]:
contact_info = """
For collaboration, contact Dr. Sarah Johnson (sarah.johnson@university.edu).
You may also reach lab assistant Mike via mike.thompson@ai-research.org or
call +1-202-555-0138. Press inquiries can be sent to press@university.edu.
"""

In [ ]:
contact_extraction_schema = {
   "name":"contact_extraction",
   "schema":{
      <TODO>
   }
}

In [ ]:
prompt = """
Please extract information from the following text:
{contact_info}
"""
response = client.chat.completions.create(
    model=model,
    messages=[{
        "role": "user",
        "content": prompt.format(contact_info=contact_info)
    }],
    temperature=0.0,
    response_format={
        "type": "json_schema",
        "json_schema": contact_extraction_schema
    }
)
message = response.choices[0].message.content
json_message = json.loads(message)
print(json.dumps(json_message, indent=2))